# Exercises XP: RAG with LangChain (Student)

## 0) Setup


In [1]:
!pip -q install -U datasets transformers sentence-transformers faiss-cpu langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface

### Option 1 (Recommended): Login programmatically with `huggingface_hub`

In [9]:
from huggingface_hub import login
from google.colab import userdata

# Get your token from Colab secrets
hf_token = userdata.get('HUGGINGFACE_KEY') # Assuming you named your secret 'HUGGINGFACE_KEY'

# Login to Hugging Face Hub
login(token=hf_token)

print("Successfully logged in to Hugging Face Hub.")

Successfully logged in to Hugging Face Hub.


### Option 2: Set as an environment variable (often picked up automatically by libraries)

In [10]:
import os
from google.colab import userdata

# Get your token from Colab secrets
hf_token = userdata.get('HUGGINGFACE_KEY') # Assuming you named your secret 'HUGGINGFACE_KEY'

# Set the environment variable
os.environ['HF_TOKEN'] = hf_token

print("HF_TOKEN environment variable set.")

HF_TOKEN environment variable set.


After running one of these cells, your API key will be configured for the current session, and Hugging Face libraries should pick it up automatically.

In [2]:
from typing import List

from datasets import load_dataset
from transformers import pipeline

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA

## 1) Load dataset and convert to Documents


In [3]:
dataset_name = "m-ric/huggingface_doc"
split = "train[:200]"
text_column = "text"
source_column = "source"

ds = load_dataset(dataset_name, split=split)

documents: List[Document] = []
for i, row in enumerate(ds):
    documents.append(
        Document(
            page_content=row[text_column],
            metadata={"source": row[source_column]}
        )
    )

print("Documents:", len(documents))
print("Example:", documents[0].metadata)
print(documents[0].page_content[:350])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Documents: 200
Example: {'source': 'huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx'}
 Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 



## 2) Split into chunks


In [4]:
chunk_size = 500 # To-Do: specify chunk size, a way to estimate is to divide average document length by desired number of chunks
chunk_overlap = 50 # To-Do: specify chunk overlap, a way to estimate is to take a small percentage of chunk size

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

splits = splitter.split_documents(documents)
print("Chunks:", len(splits))
print("First chunk:", splits[0].metadata)
print(splits[0].page_content[:350])

Chunks: 5966
First chunk: {'source': 'huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx'}
Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 




## 3) Vector store + retriever (FAISS)


In [5]:
from langchain_community.vectorstores import FAISS, DistanceStrategy

embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

vectorstore = FAISS.from_documents(
    splits,
    embeddings,
    distance_strategy=DistanceStrategy.COSINE
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Retriever ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Retriever ready


## 4) Build the RAG chain


In [22]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA  # on latest stack

llm_id = "google/flan-t5-small"

# Bypassing direct transformers.pipeline call to avoid KeyError
llm = HuggingFacePipeline(
    model_id=llm_id,
    pipeline_kwargs={"task": "text-generation"} # Changed 'task' to 'text-generation' as per transformers library's available tasks
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    return_source_documents=True # Added to return source documents
)

print("RAG chain ready")

RAG chain ready


## 5) Demo: RAG vs no-RAG


In [27]:
q = "How can I retrieve a model from the Hugging Face Hub?"

# No-RAG (LLM only)
no_rag_prompt = (
    "Answer the question. If you are not sure, say you are not sure.\n\n"
    f"Question: {q}\n"
    "Answer:"
)
no_rag_answer = hf(no_rag_prompt)[0]["generated_text"]

# RAG
rag_result = qa({"query": q}) #To-Do: fill in the question appropriately--DONE

print("Q:", q)
print("\nNo-RAG answer:\n", no_rag_answer)
print("\nRAG answer:\n", rag_result["result"])
print("\nSources:")
for d in rag_result["source_documents"]:
    print("-", d.metadata.get("source"))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How can I retrieve a model from the Hugging Face Hub?

No-RAG answer:
 Answer the question. If you are not sure, say you are not sure.

Question: How can I retrieve a model from the Hugging Face Hub?
Answer:

RAG answer:
 Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

## Joining Hugging Face and installation

To share models in the Hub, you will need to have a user. Create it on the [Hugging Face website](https://huggingface.co/join).

Now when you navigate to your Hugging Face profile, you should see your newly created model repository. Clicking on the **Files** tab will display all the files you've uploaded to the repository.

For more details on how to create and upload files to a repository, refer to the Hub documentation [here](https://huggingface.co/docs/hub/how-to-upstream).

## Upload with the web interface

This integration is made possible by the [`huggingfa

In [28]:
q = "How can I create a HuggingFace account?"

# No-RAG (LLM only)
no_rag_prompt = (
    "Answer the question. If you are not sure, say you are not sure.\n\n"
    f"Question: {q}\n"
    "Answer:"
)
no_rag_answer = hf(no_rag_prompt)[0]["generated_text"]

# RAG
rag_result = qa({"query": q})

print("Q:", q)
print("\nNo-RAG answer:\n", no_rag_answer)
print("\nRAG answer:\n", rag_result["result"])
print("\nSources:")
for d in rag_result["source_documents"]:
    print("-", d.metadata.get("source"))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How can I create a HuggingFace account?

No-RAG answer:
 Answer the question. If you are not sure, say you are not sure.

Question: How can I create a HuggingFace account?
Answer:

RAG answer:
 Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

To be able to share your model with the community, there are three more steps to follow:

1️⃣ (If it’s not already done) create an account to HF ➡ [https://huggingface.co/join](https://huggingface.co/join)

2️⃣ Sign in and store your authentication token from the Hugging Face website.

Create a new token (https://huggingface.co/settings/tokens) **with write role**

## How to use User Access Tokens?

There are plenty of ways to use a User Access Token to access the Hugging Face Hub, granting you the flexibility you need to build awesome apps on top of it.

</Tip>

## huggingface-cli login

In many cases, you must be logged in to a H

In [29]:
q = "How can I set up a HuggingFace API key?"

# No-RAG (LLM only)
no_rag_prompt = (
    "Answer the question. If you are not sure, say you are not sure.\n\n"
    f"Question: {q}\n"
    "Answer:"
)
no_rag_answer = hf(no_rag_prompt)[0]["generated_text"]

# RAG
rag_result = qa({"query": q}) #To-Do: fill in the question appropriately--DONE

print("Q:", q)
print("\nNo-RAG answer:\n", no_rag_answer)
print("\nRAG answer:\n", rag_result["result"])
print("\nSources:")
for d in rag_result["source_documents"]:
    print("-", d.metadata.get("source"))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (540 > 512). Running this sequence through the model will result in indexing errors
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How can I set up a HuggingFace API key?

No-RAG answer:
 Answer the question. If you are not sure, say you are not sure.

Question: How can I set up a HuggingFace API key?
Answer:

RAG answer:
 Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

This integration is made possible by the [`huggingface_hub`](https://github.com/huggingface/huggingface_hub) library. If you want to add your library to the Hub, we have a [guide](https://huggingface.co/docs/hub/models-adding-libraries) for you! Or simply tag someone from the Hugging Face team.

A shout out to the Hugging Face team for all the work on this integration, in particular [@osanseviero](https://twitter.com/osanseviero) 🦙.

Thank you fastlearners and hugging learners 🤗.

## How to use User Access Tokens?

There are plenty of ways to use a User Access Token to access the Hugging Face Hub, granting you the flexibility you n